# lltMemoryThroughput part 1 measurements (one T4)

Runs every part 1 step of the lab page exactly as shipped and records its output.

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import io, json, os, sys, time, contextlib, platform, subprocess
RUN = {"module": "lltMemoryThroughput", "started": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "stdout": {}, "data": {}}

class Tee(io.StringIO):
    def write(self, s):
        sys.__stdout__.write(s)
        return super().write(s)

def run_step(key, code):
    buf = Tee()
    t = time.time()
    with contextlib.redirect_stdout(buf):
        exec(code, globals())
    RUN["stdout"][key] = buf.getvalue()
    RUN.setdefault("stepSec", {})[key] = round(time.time() - t, 1)
    print(buf.getvalue())

def save(name):
    import torch
    RUN["env"] = {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
                  "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}
    RUN["finished"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    with open(f"/kaggle/working/{name}.json", "w") as f:
        json.dump(RUN, f, indent=1, default=str)
    print("saved", name)


In [ ]:
with open('gpt.py', 'w') as f:
    f.write('# gpt.py - GPT-2 small (124M) in plain PyTorch, with the two memory switches this lab measures.\nimport math\nfrom dataclasses import dataclass\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.checkpoint import checkpoint\n\n\n@dataclass\nclass GPTConfig:\n    vocab: int = 50257\n    ctx: int = 1024\n    layers: int = 12\n    heads: int = 12\n    d: int = 768\n\n\nclass Block(nn.Module):\n    def __init__(self, c):\n        super().__init__()\n        self.heads = c.heads\n        self.ln1, self.ln2 = nn.LayerNorm(c.d), nn.LayerNorm(c.d)\n        self.qkv, self.proj = nn.Linear(c.d, 3 * c.d), nn.Linear(c.d, c.d)\n        self.fc, self.out = nn.Linear(c.d, 4 * c.d), nn.Linear(4 * c.d, c.d)\n\n    def forward(self, x):\n        B, T, C = x.shape\n        q, k, v = self.qkv(self.ln1(x)).split(C, dim=2)\n        q, k, v = (t.view(B, T, self.heads, -1).transpose(1, 2) for t in (q, k, v))\n        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)\n        x = x + self.proj(y.transpose(1, 2).reshape(B, T, C))\n        return x + self.out(F.gelu(self.fc(self.ln2(x)), approximate="tanh"))\n\n\ndef chunk_loss(h, w, y):\n    return F.cross_entropy((h @ w.t()).float(), y, reduction="sum")\n\n\nclass GPT(nn.Module):\n    def __init__(self, c, ckpt=False, chunks=0):\n        super().__init__()\n        self.ckpt, self.chunks = ckpt, chunks   # activation checkpointing, chunked cross-entropy\n        self.wte, self.wpe = nn.Embedding(c.vocab, c.d), nn.Embedding(c.ctx, c.d)\n        self.blocks = nn.ModuleList([Block(c) for _ in range(c.layers)])\n        self.ln_f = nn.LayerNorm(c.d)\n        self.head = nn.Linear(c.d, c.vocab, bias=False)\n        self.head.weight = self.wte.weight          # tied, as in GPT-2\n        for n, p in self.named_parameters():\n            if p.dim() == 2:\n                small = n.endswith(("proj.weight", "out.weight"))\n                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * c.layers) if small else 0.02)\n            elif n.endswith("bias"):\n                nn.init.zeros_(p)\n\n    def forward(self, idx, targets):\n        x = self.wte(idx) + self.wpe(torch.arange(idx.shape[1], device=idx.device))\n        for b in self.blocks:\n            x = checkpoint(b, x, use_reentrant=False) if self.ckpt else b(x)\n        h, y = self.ln_f(x).flatten(0, 1), targets.flatten()\n        if not self.chunks:\n            return F.cross_entropy(self.head(h).float(), y)\n        # Chunked cross-entropy: each chunk\'s logits are recomputed in backward, so one chunk is alive at a time\n        parts = [checkpoint(chunk_loss, hc, self.head.weight, yc, use_reentrant=False)\n                 for hc, yc in zip(h.chunk(self.chunks), y.chunk(self.chunks))]\n        return sum(parts) / y.numel()\n')
print('wrote', 'gpt.py')


In [ ]:
run_step('check', '# Runtime check. Run this first. It prints the menu path that fixes each problem it finds.\nimport os, shutil, sys\nimport torch\n\nproblems = []\nif sys.version_info < (3, 11):\n    problems.append(f"Python {sys.version.split()[0]} is too old. Kaggle and Colab images ship 3.11 or later.")\nif not torch.cuda.is_available():\n    problems.append("No GPU. Kaggle: Settings > Accelerator > GPU T4 x2. Colab: Runtime > Change runtime type > T4 GPU.")\nelse:\n    names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]\n    print("GPUs:", names, "| capability", torch.cuda.get_device_capability(0))\n    if any("P100" in n for n in names):\n        problems.append("P100 found. Current PyTorch wheels dropped it. Kaggle: Settings > Accelerator > GPU T4 x2.")\n    if len(names) < 2:\n        print("One GPU. Part 2 (two GPUs) needs Kaggle: Settings > Accelerator > GPU T4 x2. Parts 1 and 3 run here.")\nram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9\ndisk = shutil.disk_usage("/tmp").free / 1e9\nprint(f"python {sys.version.split()[0]} | torch {torch.__version__} | RAM {ram:.0f} GB | free disk in /tmp {disk:.0f} GB")\nif disk < 25:\n    problems.append("Under 25 GB free in /tmp. Step 10 writes a 21 GB checkpoint there.")\nfor p in problems or ["Runtime OK"]:\n    print(p)\n')


In [ ]:
run_step('s01', 'import torch\nfrom gpt import GPT, GPTConfig\n\ncap = torch.cuda.get_device_capability()\ndtype = torch.bfloat16 if cap >= (8, 0) else torch.float16   # pick by capability, not by is_bf16_supported()\nprint(torch.cuda.get_device_name(), cap, "| is_bf16_supported():", torch.cuda.is_bf16_supported(), "| using", dtype)\n\ntorch.manual_seed(0)\nmodel = GPT(GPTConfig()).cuda()\nN = sum(p.numel() for p in model.parameters())\nopt = torch.optim.AdamW(model.parameters(), lr=6e-4, fused=True)\nscaler = torch.amp.GradScaler("cuda", enabled=dtype == torch.float16)\nx = torch.randint(0, 50257, (1, 64), device="cuda")\nfor _ in range(2):\n    with torch.autocast("cuda", dtype=dtype):\n        loss = model(x, x)\n    scaler.scale(loss).backward()      # no zero_grad: params, grads and Adam state all stay alive\n    scaler.step(opt)\n    scaler.update()\n\nGB = 1e9\nalloc = torch.cuda.memory_allocated()\nprint(f"parameters {N:,}")\nprint(f"allocated after a step {alloc / GB:.3f} GB = {alloc / N:.2f} bytes per parameter")\nprint(f"16 bytes per parameter {16 * N / GB:.3f} GB")\nprint(f"reserved by the caching allocator {torch.cuda.memory_reserved() / GB:.3f} GB")\ndel model, opt, scaler, loss, x   # loss.grad_fn still points at every parameter and its grad\ntorch.cuda.empty_cache()\n')


In [ ]:
run_step('s02', '# Peak memory of one fp16 autocast + fused AdamW step of GPT-2 124M, from the tensors autograd keeps.\nN, L, d, V = 124_439_808, 12, 768, 50257\nGB = 1e9\n\ndef predict_peak_gb(b, s, ckpt=False, chunks=0):\n    tokens = b * s\n    layer = 4 * d if ckpt else 38 * d     # bytes per token per block kept for backward\n    head = 8 * d if chunks else 6 * d     # final LayerNorm input, plus the hidden state the head reads\n    logits = 12 * V / max(chunks, 1)      # fp32 log-probs, their gradient and the logits gradient\n    during_loss = 14 * N + tokens * (L * layer + head + logits)   # params 4, Adam 8, fp16 weight copies 2\n    end_of_backward = 16 * N + 4 * V * d     # params 4, grads 4, Adam 8, plus the dense embedding gradient\n    return max(during_loss, end_of_backward) / GB\n\nfor b, s in [(1, 256), (8, 1024), (12, 1024), (16, 1024)]:\n    print(f"b={b:>2} s={s:>4}  predicted peak {predict_peak_gb(b, s):5.2f} GB")\nprint(f"per token: blocks {L * 38 * d:,} B, logits {12 * V:,} B")\n')


In [ ]:
run_step('s03', 'import gc\nimport torch\nfrom gpt import GPT, GPTConfig\n\ncfg = GPTConfig()\n\ndef train_step(model, opt, scaler, b, s, dtype=torch.float16):\n    x = torch.randint(0, cfg.vocab, (b, s), device="cuda")\n    opt.zero_grad(set_to_none=True)\n    with torch.autocast("cuda", dtype=dtype, enabled=dtype != torch.float32):\n        loss = model(x, x)\n    scaler.scale(loss).backward()\n    scaler.step(opt)\n    scaler.update()\n    return loss\n\ndef measure_peak(b, s, **switches):\n    """Peak allocated and reserved GB of the third training step, or None on out-of-memory."""\n    model = GPT(cfg, **switches).cuda()\n    opt = torch.optim.AdamW(model.parameters(), lr=6e-4, fused=True)\n    scaler = torch.amp.GradScaler("cuda")\n    result = None\n    try:\n        for i in range(3):\n            if i == 2:\n                torch.cuda.reset_peak_memory_stats()   # Adam state exists from step 1 on\n            train_step(model, opt, scaler, b, s)\n        torch.cuda.synchronize()\n        result = (torch.cuda.max_memory_allocated() / GB, torch.cuda.max_memory_reserved() / GB)\n    except torch.cuda.OutOfMemoryError:\n        pass\n    del model, opt, scaler\n    gc.collect()\n    torch.cuda.empty_cache()\n    return result\n\ngrid = []\nfor s in (256, 512, 1024):\n    for b in (1, 2, 4, 8, 12, 16):\n        pred, got = predict_peak_gb(b, s), measure_peak(b, s)\n        grid.append((b, s, pred, got))\n        if got is None:\n            print(f"b={b:>2} s={s:>4}  predicted {pred:5.2f} GB  measured OUT OF MEMORY")\n            break\n        print(f"b={b:>2} s={s:>4}  predicted {pred:5.2f}  allocated {got[0]:5.2f}  "\n              f"reserved {got[1]:5.2f} GB  error {pred / got[0] - 1:+.1%}")\n')


In [ ]:
run_step('s04', 'import time\n\ndef tokens_per_sec(b, s, steps=10, dtype=torch.float16, compile=False, **switches):\n    model = GPT(cfg, **switches).cuda()\n    if compile:\n        model = torch.compile(model)\n    opt = torch.optim.AdamW(model.parameters(), lr=6e-4, fused=True)\n    scaler = torch.amp.GradScaler("cuda", enabled=dtype == torch.float16)\n    t = time.perf_counter()\n    train_step(model, opt, scaler, b, s, dtype)      # the first step includes any compile time\n    torch.cuda.synchronize()\n    first = time.perf_counter() - t\n    for _ in range(2):\n        train_step(model, opt, scaler, b, s, dtype)  # warm-up\n    torch.cuda.synchronize()                         # wait for queued GPU work before starting the clock\n    t = time.perf_counter()\n    for _ in range(steps):\n        train_step(model, opt, scaler, b, s, dtype)\n    torch.cuda.synchronize()                         # and again before stopping it\n    tps = b * s * steps / (time.perf_counter() - t)\n    del model, opt, scaler\n    gc.collect()\n    torch.cuda.empty_cache()\n    return tps, first\n\ndef mfu(tps, s, peak=65e12):\n    flops_per_token = 6 * N + 12 * L * d * s     # 6N for the weights, plus attention scores\n    return tps * flops_per_token / peak\n\ntps, first = tokens_per_sec(8, 1024)\nprint(f"b=8 s=1024  {tps:,.0f} tokens/s  first step {first:.2f} s  MFU {mfu(tps, 1024):.1%} of 65 TFLOPS")\n\n# Timing pieces without synchronize: the clock stops when the CPU has queued the work, not when the GPU has done it\nmodel = GPT(cfg).cuda()\nx = torch.randint(0, cfg.vocab, (8, 1024), device="cuda")\nfor sync in (False, True):\n    for _ in range(2):\n        with torch.autocast("cuda", dtype=torch.float16):\n            loss = model(x, x)\n        torch.cuda.synchronize()\n        t0 = time.perf_counter()\n        with torch.autocast("cuda", dtype=torch.float16):\n            loss = model(x, x)\n        if sync:\n            torch.cuda.synchronize()\n        t1 = time.perf_counter()\n        loss.backward()\n        if sync:\n            torch.cuda.synchronize()\n        t2 = time.perf_counter()\n        model.zero_grad(set_to_none=True)\n    print(f"synchronize={sync!s:5}  forward {1e3 * (t1 - t0):6.1f} ms  backward {1e3 * (t2 - t1):6.1f} ms")\ndel model, x, loss\ngc.collect()\ntorch.cuda.empty_cache()\n')


In [ ]:
run_step('s05', 'variants = {\n    "baseline": {},\n    "checkpointing": {"ckpt": True},\n    "chunked CE x8": {"chunks": 8},\n    "both": {"ckpt": True, "chunks": 8},\n}\ncut = {}\nfor name, sw in variants.items():\n    peak = measure_peak(8, 1024, **sw)[0]\n    tps, _ = tokens_per_sec(8, 1024, **sw)\n    largest = None\n    for b in (8, 12, 16, 24, 32, 48, 64, 96, 128):\n        if measure_peak(b, 1024, **sw) is None:\n            break\n        largest = b\n    cut[name] = (peak, tps, largest)\n    print(f"{name:14} b=8 s=1024  predicted {predict_peak_gb(8, 1024, **sw):5.2f}  measured {peak:5.2f} GB  "\n          f"{tps:7,.0f} tokens/s  largest batch at s=1024: {largest}")\n')


In [ ]:
run_step('s06', 'speed = {}\nfor name, kw in [("fp32", {"dtype": torch.float32}), ("fp16 autocast", {}),\n                 ("bf16 autocast", {"dtype": torch.bfloat16}), ("fp16 + torch.compile", {"compile": True})]:\n    tps, first = tokens_per_sec(4, 1024, **kw)\n    speed[name] = (tps, first)\n    print(f"{name:22} {tps:7,.0f} tokens/s  first step {first:6.1f} s")\n\n# fp16 without loss scaling: how many gradient entries in the blocks round to exactly zero?\ntorch.manual_seed(0)\nmodel = GPT(cfg).cuda()\nx = torch.randint(0, cfg.vocab, (8, 1024), device="cuda")\nfor scale in (1.0, 65536.0):\n    model.zero_grad(set_to_none=True)\n    with torch.autocast("cuda", dtype=torch.float16):\n        loss = model(x, x)\n    (loss * scale).backward()\n    g = torch.cat([p.grad.flatten() for p in model.blocks.parameters()])\n    print(f"loss scale {scale:>7.0f}  zero gradient entries {(g == 0).float().mean().item():.2%}")\nprint("smallest fp16 above zero:", torch.finfo(torch.float16).tiny, "normal,", 2.0 ** -24, "subnormal")\ndel model, x, loss, g\ngc.collect()\ntorch.cuda.empty_cache()\n')


In [ ]:

import gc, torch
GB = 1e9
# Memory at each phase of one step, b=8 s=1024, for the account in step 2.
def phases(**sw):
    model = GPT(cfg, **sw).cuda()
    opt = torch.optim.AdamW(model.parameters(), lr=6e-4, fused=True)
    scaler = torch.amp.GradScaler("cuda")
    for _ in range(2):
        train_step(model, opt, scaler, 8, 1024)
    torch.cuda.synchronize()
    x = torch.randint(0, cfg.vocab, (8, 1024), device="cuda")
    opt.zero_grad(set_to_none=True)
    base = torch.cuda.memory_allocated()
    torch.cuda.reset_peak_memory_stats()
    with torch.autocast("cuda", dtype=torch.float16):
        loss = model(x, x)
    fwd = torch.cuda.memory_allocated(); fwd_peak = torch.cuda.max_memory_allocated()
    torch.cuda.reset_peak_memory_stats()
    scaler.scale(loss).backward()
    bwd_peak = torch.cuda.max_memory_allocated(); after_bwd = torch.cuda.memory_allocated()
    torch.cuda.reset_peak_memory_stats()
    scaler.step(opt); scaler.update(); torch.cuda.synchronize()
    step_peak = torch.cuda.max_memory_allocated()
    r = {k: round(v / GB, 3) for k, v in dict(before=base, after_forward=fwd, forward_peak=fwd_peak, backward_peak=bwd_peak,
         after_backward=after_bwd, optimizer_peak=step_peak).items()}
    r["saved_per_token_bytes"] = round((fwd - base) / 8192)
    del model, opt, scaler, x, loss; gc.collect(); torch.cuda.empty_cache()
    return r
RUN["data"]["phases"] = {name: phases(**sw) for name, sw in variants.items()}
print(json.dumps(RUN["data"]["phases"], indent=1))

# Hidden state only (no head): bytes kept per token per block.
model = GPT(cfg).cuda()
x = torch.randint(0, cfg.vocab, (8, 1024), device="cuda")
base = torch.cuda.memory_allocated()
with torch.autocast("cuda", dtype=torch.float16):
    h = model.wte(x) + model.wpe(torch.arange(1024, device="cuda"))
    for b in model.blocks:
        h = b(h)
per_block = (torch.cuda.memory_allocated() - base) / 8192 / 12
RUN["data"]["bytes_per_token_per_block"] = round(per_block, 1)
print("bytes per token per block", round(per_block, 1), "=", round(per_block / 768, 2), "x d")
del model, x, h; gc.collect(); torch.cuda.empty_cache()

# Broken run: model.half() with AdamW. eps = 1e-8 rounds to 0 in fp16.
print("1e-8 in fp16 ->", torch.tensor(1e-8, dtype=torch.float16).item())
res = {}
for mode in ("half", "autocast"):
    torch.manual_seed(0)
    model = GPT(cfg).cuda()
    if mode == "half":
        model = model.half()
    opt = torch.optim.AdamW(model.parameters(), lr=6e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=mode == "autocast")
    x = torch.randint(0, cfg.vocab, (4, 512), device="cuda", generator=torch.Generator(device="cuda").manual_seed(1))
    losses, nonfinite = [], []
    for step in range(30):
        opt.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=mode == "autocast"):
            loss = model(x, x)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update()
        losses.append(round(loss.item(), 4))
        nonfinite.append(int(sum((~torch.isfinite(p)).sum().item() for p in model.parameters())))
    res[mode] = {"losses": losses, "nonfiniteParams": nonfinite}
    print(mode, losses[:6], "...", losses[-3:], "| non-finite params after steps 1-3:", nonfinite[:3])
    del model, opt; gc.collect(); torch.cuda.empty_cache()
RUN["data"]["half_vs_autocast"] = res

# Save the numbers the steps computed.
RUN["data"]["N"] = N
RUN["data"]["grid"] = [list(g[:3]) + [list(g[3]) if g[3] else None] for g in grid]
RUN["data"]["cut"] = cut
RUN["data"]["speed"] = speed
RUN["data"]["gpu"] = torch.cuda.get_device_name(0)
save("lltMemoryThroughput_p1")
